# Notebook 01 — Gain Signal

**Continual Learning in Context: RML Extension for CL-BENCH**

Notebook 01 treats **gain** as the first measurable context signal.

Notebook 00 established the basic workflow:

\[
\text{stateful reward} - \text{stateless reward} = \text{gain}
\]

Notebook 01 asks:

> Where does gain accumulate, where does it weaken, and which variants carry the strongest context advantage?

Inputs from Notebook 00:

- `results/00_context_gain.csv`
- `results/00_context_summary.json`

Outputs from Notebook 01:

- `results/01_gain_signal_by_variant.csv`
- `results/01_gain_signal_summary.json`
- `figures/01_gain_by_variant.png`
- `figures/01_gain_cumulative.png`
- `results/01_gain_signal_artifacts.zip`

## 0. Bootstrap Repo Path

This cell works locally or in Colab.

If opened directly in Colab and the repo is missing, it clones:

```text
https://github.com/thinkthoughts/continual-learning-bench-rml
```

In [ ]:
from pathlib import Path
import sys
import os
import subprocess

REPO_URL = "https://github.com/thinkthoughts/continual-learning-bench-rml.git"
REPO_NAME = "continual-learning-bench-rml"

def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def find_rml_root(start: Path) -> Path | None:
    start = start.resolve()
    for base in [start, *start.parents]:
        if (base / "src" / "gain.py").exists():
            return base
        if (base / "rml" / "src" / "gain.py").exists():
            return base / "rml"
        if (base / REPO_NAME / "rml" / "src" / "gain.py").exists():
            return base / REPO_NAME / "rml"
    return None

cwd = Path.cwd().resolve()
RML_ROOT = find_rml_root(cwd)

if RML_ROOT is None and running_in_colab():
    repo_path = Path("/content") / REPO_NAME

    if repo_path.exists():
        print("Repo already exists. Pulling latest changes...")
        subprocess.run(["git", "-C", str(repo_path), "pull"], check=False)
    else:
        print("Cloning repo...")
        subprocess.run(["git", "clone", REPO_URL, str(repo_path)], check=True)

    RML_ROOT = repo_path / "rml"
    os.chdir(RML_ROOT)

elif RML_ROOT is not None:
    os.chdir(RML_ROOT)

else:
    raise RuntimeError(
        "Could not locate rml/src/gain.py. "
        "Run this notebook inside the repo, or in Colab allow the bootstrap cell to clone the repo."
    )

if str(RML_ROOT) not in sys.path:
    sys.path.insert(0, str(RML_ROOT))

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Running in Colab:", running_in_colab())
print("Current working directory:", Path.cwd())
print("RML root:", RML_ROOT)
print("results:", RESULTS_DIR)
print("figures:", FIGURES_DIR)

## 1. Imports

In [ ]:
import json
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.gain import compute_gain, cumulative_gain, average_gain
from src.drift import detect_drift

print("Imports complete.")

## 2. Load Notebook 00 Outputs

Notebook 01 expects Notebook 00 to have produced:

```text
results/00_context_gain.csv
results/00_context_summary.json
```

If these are missing, this notebook creates a small fallback trajectory so the notebook remains runnable.

In [ ]:
input_csv = RESULTS_DIR / "00_context_gain.csv"
input_json = RESULTS_DIR / "00_context_summary.json"

if input_csv.exists():
    df = pd.read_csv(input_csv)
    print("Loaded:", input_csv)
else:
    print("Notebook 00 CSV not found. Creating fallback toy trajectory.")

    instances = np.arange(1, 13)
    df = pd.DataFrame({
        "instance": instances,
        "variant": [
            "A", "A", "A", "A",
            "B", "B", "B", "B",
            "C", "C", "C", "C",
        ],
        "stateful_reward": [
            0.18, 0.22, 0.28, 0.35,
            0.43, 0.48, 0.46, 0.52,
            0.40, 0.45, 0.55, 0.60,
        ],
        "stateless_reward": [
            0.18, 0.19, 0.20, 0.21,
            0.22, 0.20, 0.21, 0.22,
            0.23, 0.22, 0.24, 0.25,
        ],
    })
    df["gain"] = compute_gain(
        df["stateful_reward"].tolist(),
        df["stateless_reward"].tolist(),
    )

if "gain" not in df.columns:
    df["gain"] = compute_gain(
        df["stateful_reward"].tolist(),
        df["stateless_reward"].tolist(),
    )

df

## 3. Gain as Context Signal

Gain is the measurable advantage of prior experience:

\[
g_t = r_t^{sf} - r_t^{sl}
\]

In this notebook:

- positive gain means accumulated context helped
- zero gain means context added no measurable advantage
- negative gain suggests stale or harmful context

In [ ]:
df["cumulative_gain"] = df["gain"].cumsum()
df["gain_delta"] = df["gain"].diff().fillna(0.0)

df["gain_sign"] = np.where(
    df["gain"] > 0,
    "positive",
    np.where(df["gain"] < 0, "negative", "zero")
)

df

## 4. Variant-Level Gain

Each variant represents a local context region.

The question is:

> Which variant produces the strongest reusable context?

In [ ]:
variant_summary = (
    df.groupby("variant")
    .agg(
        instances=("instance", "count"),
        mean_gain=("gain", "mean"),
        total_gain=("gain", "sum"),
        min_gain=("gain", "min"),
        max_gain=("gain", "max"),
        mean_stateful_reward=("stateful_reward", "mean"),
        mean_stateless_reward=("stateless_reward", "mean"),
    )
    .reset_index()
)

variant_summary

## 5. Gain Growth

This section checks whether gain is accumulating over the schedule.

A positive final cumulative gain means context improved the system over repeated experience.

In [ ]:
gain_summary = {
    "total_instances": int(len(df)),
    "positive_gain_instances": int((df["gain"] > 0).sum()),
    "zero_gain_instances": int((df["gain"] == 0).sum()),
    "negative_gain_instances": int((df["gain"] < 0).sum()),
    "cumulative_gain": float(df["gain"].sum()),
    "average_gain": float(df["gain"].mean()),
    "final_cumulative_gain": float(df["cumulative_gain"].iloc[-1]),
    "largest_gain": float(df["gain"].max()),
    "largest_gain_instance": int(df.loc[df["gain"].idxmax(), "instance"]),
    "largest_gain_drop": float(df["gain_delta"].min()),
    "largest_gain_drop_instance": int(df.loc[df["gain_delta"].idxmin(), "instance"]),
}

gain_summary

## 6. Stale-Context Candidates

Negative gain is the clearest warning signal.

Large drops in gain can also indicate that the system is entering a new context region where prior experience is less useful.

In [ ]:
# Direct negative-gain flags.
negative_gain_indices = df.index[df["gain"] < 0].tolist()

# Drop flags: strongest negative gain deltas.
drop_threshold = -0.08
drop_indices = df.index[df["gain_delta"] <= drop_threshold].tolist()

# Use existing drift helper for negative gain under threshold.
drift_indices = detect_drift(df["gain"].tolist(), threshold=-0.02)

df["negative_gain_flag"] = False
df.loc[negative_gain_indices, "negative_gain_flag"] = True

df["gain_drop_flag"] = False
df.loc[drop_indices, "gain_drop_flag"] = True

df["drift_flag"] = False
df.loc[drift_indices, "drift_flag"] = True

df[[
    "instance",
    "variant",
    "gain",
    "gain_delta",
    "negative_gain_flag",
    "gain_drop_flag",
    "drift_flag",
]]

## 7. Figure — Gain by Variant

This figure summarizes where context advantage is strongest.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    variant_summary["variant"],
    variant_summary["mean_gain"],
)

ax.axhline(0, linewidth=1)
ax.set_title("Notebook 01: Mean Gain by Variant")
ax.set_xlabel("Variant")
ax.set_ylabel("Mean Gain")
ax.grid(True, axis="y", alpha=0.3)

gain_by_variant_path = FIGURES_DIR / "01_gain_by_variant.png"
fig.tight_layout()
fig.savefig(gain_by_variant_path, dpi=160)

gain_by_variant_path

## 8. Figure — Cumulative Gain

Cumulative gain shows whether context value accumulates, stalls, or reverses over the schedule.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    df["instance"],
    df["cumulative_gain"],
    marker="o",
)

for boundary in df.groupby("variant")["instance"].min().tolist()[1:]:
    ax.axvline(boundary - 0.5, linestyle=":", linewidth=1)

ax.axhline(0, linewidth=1)
ax.set_title("Notebook 01: Cumulative Gain")
ax.set_xlabel("Instance")
ax.set_ylabel("Cumulative Gain")
ax.grid(True, alpha=0.3)

cumulative_gain_path = FIGURES_DIR / "01_gain_cumulative.png"
fig.tight_layout()
fig.savefig(cumulative_gain_path, dpi=160)

cumulative_gain_path

## 9. Export Artifacts

In [ ]:
variant_csv_path = RESULTS_DIR / "01_gain_signal_by_variant.csv"
summary_json_path = RESULTS_DIR / "01_gain_signal_summary.json"
trajectory_csv_path = RESULTS_DIR / "01_gain_signal_trajectory.csv"
zip_path = RESULTS_DIR / "01_gain_signal_artifacts.zip"

variant_summary.to_csv(variant_csv_path, index=False)
df.to_csv(trajectory_csv_path, index=False)

summary_with_metadata = {
    **gain_summary,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "01_gain_signal.ipynb",
    "extension": "continual-learning-bench-rml",
    "input_csv": str(input_csv),
    "repo": REPO_URL,
}

with open(summary_json_path, "w") as f:
    json.dump(summary_with_metadata, f, indent=2)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in [
        variant_csv_path,
        trajectory_csv_path,
        summary_json_path,
        gain_by_variant_path,
        cumulative_gain_path,
    ]:
        z.write(path, arcname=path.name)

print("Saved artifacts:")
for path in [
    variant_csv_path,
    trajectory_csv_path,
    summary_json_path,
    gain_by_variant_path,
    cumulative_gain_path,
    zip_path,
]:
    print("-", path)

## 10. Download Artifacts

In Colab, the next cell downloads the zip.

Locally, the archive is saved under:

```text
rml/results/01_gain_signal_artifacts.zip
```

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Download helper is only active in Colab.")
    print("Artifact archive:", zip_path)

## 11. Notebook 01 Claim

Notebook 01 defines gain as the first observable signal of useful context.

\[
\text{gain} > 0
\quad \Rightarrow \quad
\text{prior experience improved performance}
\]

In RML terms:

> Gain is context made measurable.

Next notebook:

```text
07_latent_structure.ipynb
```

will ask what structure the system appears to be learning.